# Lesson 11: Flair NER - Contextual String Embeddings

## Overview

**Flair** is a powerful NLP framework developed by Zalando Research that introduced the concept of **contextual string embeddings** - character-level embeddings that capture context from the entire sentence. Flair achieves state-of-the-art results on NER tasks while maintaining a clean, simple API.

### What Makes Flair Special?

| Feature | Description |
|---------|-------------|
| **Contextual String Embeddings** | Character-level LM embeddings that model words in context |
| **Embedding Stacking** | Combine multiple embedding types (Flair + GloVe + BERT) |
| **Pre-trained Models** | Extensive model zoo for 12+ languages |
| **Simple API** | Clean Pythonic interface for training and inference |
| **Active Maintenance** | Continuously updated with latest research |

### Key Innovations

1. **Character-level Language Models**: Instead of using pre-computed word embeddings, Flair uses the internal states of character-level LMs as word representations

2. **Contextual Understanding**: The same word gets different embeddings based on its surrounding context

3. **Subword Modeling**: Naturally handles out-of-vocabulary words and morphologically rich languages

### Learning Objectives

By the end of this lesson, you will:
- Understand how Flair's contextual string embeddings work
- Use pre-trained Flair NER models
- Stack different embedding types for improved performance
- Fine-tune Flair NER on custom datasets
- Compare Flair with BERT-based approaches

### References

- **Paper**: [Contextual String Embeddings for Sequence Labeling](https://aclanthology.org/C18-1139/) (Akbik et al., 2018)
- **GitHub**: [https://github.com/flairNLP/flair](https://github.com/flairNLP/flair)
- **Documentation**: [https://flairnlp.github.io/](https://flairnlp.github.io/)
- **Model Hub**: [https://huggingface.co/flair](https://huggingface.co/flair)

## 1. Environment Setup

In [ ]:
# Install Flair and dependencies
!pip install -q flair>=0.13 seqeval datasets

print("Installation complete!")

In [ ]:
import flair
from flair.data import Sentence, Corpus
from flair.models import SequenceTagger
from flair.embeddings import (
    TokenEmbeddings, 
    WordEmbeddings, 
    FlairEmbeddings, 
    StackedEmbeddings,
    TransformerWordEmbeddings
)
from flair.trainers import ModelTrainer
from flair.datasets import ColumnCorpus
import torch
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

print(f"Flair version: {flair.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Understanding Contextual String Embeddings

### 2.1 The Core Innovation

Traditional word embeddings (Word2Vec, GloVe) give the same vector for a word regardless of context:

```
"I put my money in the bank"  →  bank = [0.2, 0.5, ...] (same)
"I sat on the river bank"     →  bank = [0.2, 0.5, ...] (same)
```

Flair uses character-level language models to produce **contextualized** embeddings:

```
"I put my money in the bank"  →  bank = [0.2, 0.5, ...] (financial)
"I sat on the river bank"     →  bank = [0.8, 0.1, ...] (geographical)
```

### 2.2 Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                 FLAIR CONTEXTUAL STRING EMBEDDINGS                   │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  Input: "Barack Obama visited Paris"                               │
│          ↓                                                          │
│  Character Sequence: B-a-r-a-c-k- -O-b-a-m-a- -v-i-s-i-t-e-d...    │
│          ↓                                                          │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │     Forward Character LM (reads left-to-right)              │   │
│  │     → Captures left context for each character              │   │
│  └─────────────────────────────────────────────────────────────┘   │
│          ↓                                                          │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │     Backward Character LM (reads right-to-left)             │   │
│  │     → Captures right context for each character             │   │
│  └─────────────────────────────────────────────────────────────┘   │
│          ↓                                                          │
│  Extract hidden states at word boundaries:                         │
│  - Forward LM state after last character of word                   │
│  - Backward LM state before first character of word                │
│          ↓                                                          │
│  Word Embedding = [forward_state ; backward_state]                 │
│                                                                     │
│  "Barack" → [h_fwd_k ; h_bwd_B]                                    │
│  "Obama"  → [h_fwd_a ; h_bwd_O]                                    │
│  etc.                                                              │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

## 3. Using Pre-trained Flair NER Models

Flair provides excellent pre-trained NER models for multiple languages.

In [ ]:
# Load the best English NER model
# ner-large is trained on CoNLL-03 with Flair + GloVe + BERT embeddings
tagger = SequenceTagger.load('ner')

print(f"Model loaded!")
print(f"Tag dictionary: {tagger.tag_dictionary.get_items()}")

In [ ]:
# Basic NER tagging
text = "Barack Obama was born in Hawaii and served as the 44th President of the United States."

# Create a Flair Sentence object
sentence = Sentence(text)

# Predict NER tags
tagger.predict(sentence)

# Display results
print("INPUT TEXT:")
print(text)
print("\nENTITIES FOUND:")
for entity in sentence.get_spans('ner'):
    print(f"  {entity.text:25} → {entity.tag:10} (confidence: {entity.score:.4f})")

# Show the tagged sentence
print(f"\nTAGGED SENTENCE:")
print(sentence.to_tagged_string())

In [ ]:
# Access detailed token-level information
print("TOKEN-LEVEL DETAILS:")
print("-" * 60)
for token in sentence:
    # Get the NER tag for this token
    tag = token.get_label('ner')
    if tag.value != 'O':
        print(f"Token: {token.text:15} | Tag: {tag.value:10} | Score: {tag.score:.4f}")

### 3.1 Available Pre-trained Models

Flair offers multiple NER models with different speed/accuracy tradeoffs:

In [ ]:
# Available English NER models
english_models = {
    'ner': 'Standard 4-class NER (PER, LOC, ORG, MISC) - balanced',
    'ner-fast': 'Faster model with slightly lower accuracy',
    'ner-large': 'Highest accuracy, uses Flair + GloVe + BERT',
    'ner-ontonotes': '18-class NER trained on OntoNotes',
    'ner-ontonotes-large': 'Large 18-class model'
}

print("AVAILABLE ENGLISH NER MODELS:")
print("="*70)
for model_name, description in english_models.items():
    print(f"  {model_name:25} - {description}")

# Multilingual models
print("\nMULTILINGUAL MODELS:")
print("="*70)
multilingual = ['de-ner', 'fr-ner', 'nl-ner', 'da-ner', 'ar-ner', 'ner-multi']
for model in multilingual:
    print(f"  {model}")

In [ ]:
# Test with OntoNotes model (18 entity types)
onto_tagger = SequenceTagger.load('ner-ontonotes')

onto_text = """
Apple Inc. reported earnings of $20 billion for Q3 2023. 
CEO Tim Cook announced the results at 2 PM Pacific Time.
The stock rose 5% on the New York Stock Exchange.
"""

sentence = Sentence(onto_text)
onto_tagger.predict(sentence)

print("ONTONOTES NER (18 ENTITY TYPES):")
print("="*70)
for entity in sentence.get_spans('ner'):
    print(f"  {entity.text:30} → {entity.tag:15} ({entity.score:.3f})")

# OntoNotes entity types
print("\nOntoNotes Entity Types:")
onto_types = [
    "PERSON", "NORP", "FAC", "ORG", "GPE", "LOC", "PRODUCT", "EVENT",
    "WORK_OF_ART", "LAW", "LANGUAGE", "DATE", "TIME", "PERCENT",
    "MONEY", "QUANTITY", "ORDINAL", "CARDINAL"
]
print(f"  {', '.join(onto_types)}")

## 4. Understanding Flair Embeddings

### 4.1 Types of Embeddings in Flair

In [ ]:
# 1. Classic Word Embeddings (GloVe, FastText)
glove_embedding = WordEmbeddings('glove')

# 2. Flair Embeddings (Contextual String Embeddings)
flair_forward = FlairEmbeddings('news-forward')
flair_backward = FlairEmbeddings('news-backward')

# 3. Transformer Embeddings (BERT, RoBERTa, etc.)
# bert_embedding = TransformerWordEmbeddings('bert-base-uncased')

print("Embedding types loaded!")
print(f"GloVe dimension: {glove_embedding.embedding_length}")
print(f"Flair forward dimension: {flair_forward.embedding_length}")
print(f"Flair backward dimension: {flair_backward.embedding_length}")

In [ ]:
# Demonstrate contextual vs static embeddings
sentence1 = Sentence("I deposited money in the bank.")
sentence2 = Sentence("I sat on the river bank.")

# Embed with GloVe (static - same for "bank" in both)
glove_embedding.embed([sentence1, sentence2])

# Get embeddings for "bank" in each sentence
bank1_glove = sentence1[6].embedding  # "bank" in sentence1
bank2_glove = sentence2[5].embedding  # "bank" in sentence2

# Calculate cosine similarity
from torch.nn.functional import cosine_similarity

glove_sim = cosine_similarity(bank1_glove.unsqueeze(0), bank2_glove.unsqueeze(0)).item()
print(f"GloVe similarity for 'bank' in different contexts: {glove_sim:.4f}")
print("(Should be ~1.0 since GloVe is context-independent)")

In [ ]:
# Now with Flair embeddings (contextual)
sentence1 = Sentence("I deposited money in the bank.")
sentence2 = Sentence("I sat on the river bank.")

# Use stacked Flair embeddings
stacked_flair = StackedEmbeddings([flair_forward, flair_backward])
stacked_flair.embed([sentence1, sentence2])

# Get embeddings for "bank"
bank1_flair = sentence1[6].embedding
bank2_flair = sentence2[5].embedding

flair_sim = cosine_similarity(bank1_flair.unsqueeze(0), bank2_flair.unsqueeze(0)).item()
print(f"Flair similarity for 'bank' in different contexts: {flair_sim:.4f}")
print("(Lower similarity shows Flair captures different meanings!)")

### 4.2 Stacking Embeddings for Better Performance

Flair's power comes from combining multiple embedding types:

In [ ]:
# Create a powerful stacked embedding
# This is similar to what the 'ner-large' model uses
stacked_embeddings = StackedEmbeddings([
    WordEmbeddings('glove'),           # Static word embeddings
    FlairEmbeddings('news-forward'),   # Forward contextual
    FlairEmbeddings('news-backward'),  # Backward contextual
])

print(f"Stacked embedding dimension: {stacked_embeddings.embedding_length}")
print("\nComponents:")
print(f"  - GloVe: 100 dimensions")
print(f"  - Flair forward: 2048 dimensions")
print(f"  - Flair backward: 2048 dimensions")
print(f"  - Total: 4196 dimensions")

## 5. Batch Processing and Efficiency

In [ ]:
import time

# Create multiple sentences
texts = [
    "Apple announced a new iPhone at their Cupertino headquarters.",
    "Google's CEO Sundar Pichai met with European regulators in Brussels.",
    "Microsoft acquired Activision Blizzard for $69 billion.",
    "Elon Musk's SpaceX launched another Starlink satellite.",
    "Amazon opened a new fulfillment center in Texas."
]

sentences = [Sentence(text) for text in texts]

# Batch prediction
start_time = time.time()
tagger.predict(sentences, mini_batch_size=32)
batch_time = time.time() - start_time

print(f"Batch processing time for {len(texts)} sentences: {batch_time:.2f}s")
print(f"Average per sentence: {batch_time/len(texts)*1000:.1f}ms")

# Display results
print("\nEXTRACTED ENTITIES:")
print("="*70)
for sent in sentences:
    print(f"\nText: {sent.to_original_text()[:60]}...")
    for entity in sent.get_spans('ner'):
        print(f"  → {entity.text} [{entity.tag}]")

## 6. Fine-tuning Flair NER on Custom Data

### 6.1 Preparing the Dataset

In [ ]:
import os

# Create sample training data in CoNLL format
train_data = """Apple B-ORG
announced O
the O
new O
iPhone B-PRODUCT
15 I-PRODUCT
Pro I-PRODUCT
. O

Tim B-PER
Cook I-PER
is O
the O
CEO O
of O
Apple B-ORG
. O

Microsoft B-ORG
released O
Windows B-PRODUCT
11 I-PRODUCT
in O
Redmond B-LOC
. O

Satya B-PER
Nadella I-PER
leads O
Microsoft B-ORG
. O

"""

test_data = """Google B-ORG
launched O
Pixel B-PRODUCT
8 I-PRODUCT
. O

Sundar B-PER
Pichai I-PER
works O
at O
Google B-ORG
. O

"""

# Create data directory
os.makedirs('flair_data', exist_ok=True)

# Save train and test files
with open('flair_data/train.txt', 'w') as f:
    f.write(train_data)
with open('flair_data/test.txt', 'w') as f:
    f.write(test_data)
with open('flair_data/dev.txt', 'w') as f:
    f.write(test_data)  # Use same as test for this demo

print("Training data created in CoNLL format!")

In [ ]:
# Load the corpus
columns = {0: 'text', 1: 'ner'}

corpus: Corpus = ColumnCorpus(
    'flair_data',
    columns,
    train_file='train.txt',
    test_file='test.txt',
    dev_file='dev.txt'
)

print(f"Train sentences: {len(corpus.train)}")
print(f"Dev sentences: {len(corpus.dev)}")
print(f"Test sentences: {len(corpus.test)}")
print(f"\nTag dictionary: {corpus.make_label_dictionary(label_type='ner').get_items()}")

### 6.2 Building the Tagger Model

In [ ]:
# Create tag dictionary from corpus
label_dict = corpus.make_label_dictionary(label_type='ner')

# Create embeddings (using smaller embeddings for demo)
embedding_types = [
    WordEmbeddings('glove'),
    FlairEmbeddings('news-forward-fast'),
    FlairEmbeddings('news-backward-fast'),
]

embeddings = StackedEmbeddings(embeddings=embedding_types)

# Build the sequence tagger
custom_tagger = SequenceTagger(
    hidden_size=256,
    embeddings=embeddings,
    tag_dictionary=label_dict,
    tag_type='ner',
    use_crf=True  # Use CRF for sequence labeling
)

print(f"Model created with {sum(p.numel() for p in custom_tagger.parameters())/1e6:.1f}M parameters")

In [ ]:
# Train the model (using minimal epochs for demo)
trainer = ModelTrainer(custom_tagger, corpus)

# For production, use more epochs (150+) and learning rate annealing
trainer.train(
    'flair_model',
    learning_rate=0.1,
    mini_batch_size=32,
    max_epochs=3,  # Use 150+ for real training
    embeddings_storage_mode='none'  # Save memory
)

print("\nTraining complete!")

### 6.3 Using Transformer Embeddings

In [ ]:
# For best results, use transformer embeddings
# This combines BERT with Flair embeddings

# Note: This requires more GPU memory
# transformer_embeddings = StackedEmbeddings([
#     TransformerWordEmbeddings('bert-base-uncased', layers='-1'),
#     FlairEmbeddings('news-forward'),
#     FlairEmbeddings('news-backward'),
# ])

print("Transformer + Flair combination achieves highest performance!")
print("\nRecommended embeddings for production:")
print("  1. bert-base-uncased (or roberta-base)")
print("  2. news-forward (Flair forward LM)")
print("  3. news-backward (Flair backward LM)")

## 7. Comparison with Other NER Approaches

In [ ]:
import pandas as pd

# Performance comparison on CoNLL-2003
comparison_data = {
    "Model": [
        "BiLSTM-CRF",
        "ELMo",
        "BERT-base",
        "Flair (forward+backward)",
        "Flair + GloVe",
        "Flair + GloVe + ELMo",
        "Flair + GloVe + BERT",
        "SpanMarker",
        "RoBERTa-large"
    ],
    "F1 Score": [91.21, 92.22, 92.40, 92.61, 92.86, 93.09, 93.18, 93.10, 93.22],
    "Year": [2016, 2018, 2019, 2018, 2018, 2018, 2019, 2023, 2019],
    "Embeddings": [
        "Word2Vec",
        "ELMo",
        "BERT",
        "Flair",
        "Flair + GloVe",
        "Flair + GloVe + ELMo",
        "Flair + GloVe + BERT",
        "Encoder + Span",
        "RoBERTa"
    ]
}

df = pd.DataFrame(comparison_data)
print("CoNLL-2003 NER PERFORMANCE COMPARISON")
print("="*70)
print(df.to_string(index=False))
print("\nKey insight: Stacking embeddings (Flair's approach) often outperforms single embeddings!")

## 8. Advanced Features

### 8.1 Multi-task Learning

In [ ]:
# Flair supports multiple taggers on the same sentence
from flair.models import SequenceTagger

# Load NER and POS taggers
ner_tagger = SequenceTagger.load('ner')
pos_tagger = SequenceTagger.load('pos')  # Part-of-speech tagger

sentence = Sentence("Apple CEO Tim Cook announced the new iPhone.")

# Apply both taggers
ner_tagger.predict(sentence)
pos_tagger.predict(sentence)

print("MULTI-TASK OUTPUT:")
print("="*70)
print(f"\nOriginal: {sentence.to_original_text()}")
print(f"\nWith NER: {sentence.to_tagged_string('ner')}")
print(f"\nWith POS: {sentence.to_tagged_string('pos')}")

# Access both labels for each token
print("\nToken-level multi-task labels:")
for token in sentence:
    ner = token.get_label('ner').value
    pos = token.get_label('pos').value
    if ner != 'O':
        print(f"  {token.text}: NER={ner}, POS={pos}")

### 8.2 Document-level NER

In [ ]:
def process_document(text: str, tagger) -> List[Dict]:
    """
    Process a multi-sentence document and return all entities.
    """
    from flair.tokenization import SegtokSentenceSplitter
    
    # Split document into sentences
    splitter = SegtokSentenceSplitter()
    sentences = splitter.split(text)
    
    # Batch predict
    tagger.predict(sentences)
    
    # Collect all entities with their positions
    all_entities = []
    char_offset = 0
    
    for sent in sentences:
        for entity in sent.get_spans('ner'):
            all_entities.append({
                'text': entity.text,
                'type': entity.tag,
                'score': entity.score,
                'start': entity.start_position + char_offset,
                'end': entity.end_position + char_offset,
                'sentence': sent.to_original_text()
            })
        char_offset += len(sent.to_original_text()) + 1  # +1 for space
    
    return all_entities

# Test on a document
document = """
Apple Inc. reported strong quarterly earnings yesterday. CEO Tim Cook 
attributed the success to iPhone sales. The company's stock rose 5% on 
the New York Stock Exchange. Analysts at Goldman Sachs raised their price 
target to $200.
"""

entities = process_document(document, tagger)

print("DOCUMENT-LEVEL NER:")
print("="*70)
for e in entities:
    print(f"  {e['text']:25} | {e['type']:8} | pos: {e['start']}-{e['end']}")

### 8.3 Entity Linking Integration

In [ ]:
# Flair supports entity linking (connecting entities to knowledge bases)
# Note: Requires additional model download

# from flair.models import EntityLinker
# linker = EntityLinker.load('wikipedia')

# sentence = Sentence("Barack Obama was born in Hawaii.")
# tagger.predict(sentence)
# linker.predict(sentence)

# for entity in sentence.get_spans('ner'):
#     print(f"{entity.text} -> {entity.get_label('link').value}")

print("Entity Linking connects extracted entities to Wikipedia/Wikidata!")
print("\nExample output:")
print("  Barack Obama -> Q76 (Wikidata ID)")
print("  Hawaii -> Q782 (Wikidata ID)")

## 9. Production Deployment

In [ ]:
from dataclasses import dataclass
from typing import List, Optional
import time

@dataclass
class NEREntity:
    """Structured NER entity result."""
    text: str
    label: str
    score: float
    start: int
    end: int

class FlairNERService:
    """
    Production-ready Flair NER service.
    
    Features:
    - Batch processing for efficiency
    - GPU acceleration when available
    - Configurable model selection
    - Timing statistics
    """
    
    def __init__(
        self, 
        model_name: str = 'ner',
        batch_size: int = 32,
        use_gpu: bool = True
    ):
        self.batch_size = batch_size
        
        # Set device
        if use_gpu and torch.cuda.is_available():
            flair.device = torch.device('cuda:0')
        else:
            flair.device = torch.device('cpu')
        
        # Load model
        self.tagger = SequenceTagger.load(model_name)
        self._stats = {'requests': 0, 'entities': 0, 'total_time': 0}
    
    def predict(self, text: str) -> List[NEREntity]:
        """Predict entities in a single text."""
        return self.predict_batch([text])[0]
    
    def predict_batch(self, texts: List[str]) -> List[List[NEREntity]]:
        """Predict entities in multiple texts efficiently."""
        start_time = time.time()
        
        # Create sentences
        sentences = [Sentence(text) for text in texts]
        
        # Batch predict
        self.tagger.predict(
            sentences, 
            mini_batch_size=self.batch_size,
            verbose=False
        )
        
        # Extract results
        results = []
        for sent in sentences:
            entities = []
            for span in sent.get_spans('ner'):
                entities.append(NEREntity(
                    text=span.text,
                    label=span.tag,
                    score=span.score,
                    start=span.start_position,
                    end=span.end_position
                ))
            results.append(entities)
            self._stats['entities'] += len(entities)
        
        # Update stats
        self._stats['requests'] += len(texts)
        self._stats['total_time'] += time.time() - start_time
        
        return results
    
    def get_stats(self) -> dict:
        """Get service statistics."""
        return {
            **self._stats,
            'avg_time_per_request': self._stats['total_time'] / max(1, self._stats['requests']),
            'device': str(flair.device)
        }

# Initialize and test service
service = FlairNERService(model_name='ner', batch_size=32)

test_texts = [
    "Apple announced new products at their headquarters.",
    "Elon Musk visited the Tesla factory in Berlin.",
    "The United Nations held a meeting in Geneva."
]

results = service.predict_batch(test_texts)

print("FLAIR NER SERVICE TEST:")
print("="*70)
for text, entities in zip(test_texts, results):
    print(f"\nText: {text}")
    for e in entities:
        print(f"  → {e.text} [{e.label}] (score: {e.score:.3f})")

print(f"\nService Stats: {service.get_stats()}")

## 10. Best Practices and Tips

### 10.1 Model Selection Guide

In [ ]:
model_guide = """
FLAIR MODEL SELECTION GUIDE
============================

1. FOR SPEED (Production with high throughput):
   → Use 'ner-fast' or 'ner'
   → Consider smaller batch sizes for memory efficiency

2. FOR ACCURACY (When quality is paramount):
   → Use 'ner-large' (stacked Flair + GloVe + BERT)
   → Fine-tune with TransformerWordEmbeddings

3. FOR FINE-GRAINED ENTITIES (18 types):
   → Use 'ner-ontonotes' or 'ner-ontonotes-large'
   → Includes DATE, TIME, MONEY, PERCENT, etc.

4. FOR MULTILINGUAL:
   → Use 'ner-multi' (covers many languages)
   → Or language-specific: 'de-ner', 'fr-ner', etc.

5. FOR CUSTOM DOMAINS:
   → Start with pre-trained model
   → Fine-tune with domain-specific data
   → Use stacked embeddings for best results
"""
print(model_guide)

In [ ]:
# Memory optimization tips
optimization_tips = """
MEMORY OPTIMIZATION TIPS
========================

1. Use embeddings_storage_mode='none' during training:
   trainer.train(..., embeddings_storage_mode='none')
   
2. Use mini_batch_size appropriate for your GPU:
   - 4GB VRAM: batch_size=8-16
   - 8GB VRAM: batch_size=16-32
   - 16GB+ VRAM: batch_size=32-64

3. For inference, clear embeddings after prediction:
   for token in sentence:
       token.clear_embeddings()

4. Use 'fast' variants for reduced memory:
   FlairEmbeddings('news-forward-fast')
   
5. Consider CPU inference for production:
   flair.device = torch.device('cpu')
"""
print(optimization_tips)

## 11. Summary and Key Takeaways

### What We Learned

1. **Contextual String Embeddings**: Flair's character-level LM approach captures context-dependent word meanings

2. **Embedding Stacking**: Combining multiple embedding types (Flair + GloVe + BERT) achieves best performance

3. **Pre-trained Models**: Extensive model zoo for multiple languages and entity type granularities

4. **Clean API**: Simple Pythonic interface for both inference and training

### When to Use Flair

| Use Case | Recommendation |
|----------|----------------|
| Need highest accuracy | Flair + BERT stacking |
| Multilingual NER | Flair multilingual models |
| Limited labeled data | Pre-trained Flair embeddings + fine-tuning |
| Production with speed | Flair-fast or standard |
| Research/experiments | Full Flair with all embeddings |

### Further Reading

- [Flair Documentation](https://flairnlp.github.io/)
- [Contextual String Embeddings Paper](https://aclanthology.org/C18-1139/)
- [FLERT: Document-Level Features Paper](https://arxiv.org/abs/2011.06993)
- [Flair GitHub Repository](https://github.com/flairNLP/flair)

## 12. Exercises

### Exercise 1: Multi-Lingual NER
Load a German NER model and extract entities from German text.

### Exercise 2: Custom Embeddings
Create a stacked embedding with BERT + Flair and compare performance.

### Exercise 3: Domain Adaptation
Fine-tune the NER model on a domain-specific dataset (e.g., biomedical).

### Exercise 4: Production API
Build a FastAPI service that exposes the Flair NER model.

In [ ]:
# Exercise 1 Starter: German NER
# german_tagger = SequenceTagger.load('de-ner')
# german_text = "Angela Merkel besuchte Berlin gestern."
# sentence = Sentence(german_text)
# german_tagger.predict(sentence)
# print(sentence.to_tagged_string())

print("Exercise starter code ready!")
print("Uncomment and run to test German NER.")

In [ ]:
# Cleanup
import shutil
if os.path.exists('flair_data'):
    shutil.rmtree('flair_data')
if os.path.exists('flair_model'):
    shutil.rmtree('flair_model')

print("\n" + "="*70)
print("Lesson 11 Complete: Flair NER")
print("="*70)
print("""
You've learned:
- How contextual string embeddings work
- Using pre-trained Flair NER models
- Stacking embeddings for better performance
- Fine-tuning on custom datasets
- Production deployment patterns
- Multi-task NER with POS tagging

Congratulations on completing the NER Tutorial Series!
""")